# Lab 1: Fundamentals of Image Processing with OpenCV, NumPy, and Matplotlib

Welcome to your first computer vision lab session. The primary goal of this session is to build a deep, intuitive understanding of **how digital images are represented in memory**, how data flows through processing pipelines, and how mathematical arrays translate into visual pixels.

Rather than treating functions like black boxes, we will focus heavily on data structures, array dimensions, underlying primitives, and numerical limitations.

In [ ]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
# run this code to import the three primary libraries 
# (make sure a green checkmark appears, this should not take more than a few seconds)
import cv2
import numpy as np
import matplotlib.pyplot as plt

## Section 1: Core Array Structures & Digital Image Representations

### 1.1 In-Depth: Array Representation, Dimensions, Data Types & I/O

When a digital image is loaded into Python using **OpenCV (`cv2`)**, it is stored natively as a **NumPy N-dimensional array (`ndarray`)**. This is the fundamental data structure that allows us to manipulate images as numerical arrays and it is very important to understand the underlying structure of this data type thouroughly.

#### 1. Dimension Order:
- **Color Images:** Stored as a 3D array with the layout **`(Height, Width, Channels)`** or **`(H, W, C)`**.
  - `H` represents the vertical rows (y-axis).
  - `W` represents the horizontal columns (x-axis).
  - `C` represents the distinct color components.
- **Grayscale Images:** Stored as a **2D array** with the layout **`(Height, Width)`** or **`(H, W)`**. Notice that the channel dimension drops entirely because it is technically not needed. It does *not* become `(H, W, 1)`. This is a critical structural difference when designing algorithms that process both types. 

At any point during programming, it is important to know what each dimension of your array represents! it might therefore be beneficial to add this additional dimension again so you know what dimension the channel dimension is (so `(H, W, 1)`). When working with video, a temporal dimension is also added to the array, usually represented by `T` and added to the front of the array (e.g., `(T, H, W, C)`). when representing a stack of data samples (can be images or videos), another dimension can be added to the front of the array to represent the stack dimension (e.g., `(B, T, H, W, C)`).

#### 2. Native OpenCV Channel Order (BGR) vs. Matplotlib (RGB):
- Historically, OpenCV defaults to reading color images in the **BGR (Blue, Green, Red)** sequence instead of the conventional **RGB** standard. Failing to account for this will result in inverted colors when visualizing with standard plotting libraries. When working with OpenCV, it is important to be aware of this channel order and know at each step whether you are working with BGR or RGB! It can be convenient to convert to RGB right after loading the image, but keep in mind that when storing the image back to disk (using `cv2.imwrite()`), you will first need to convert the image back to BGR.

#### 3. Data Types & The Danger of Integer Overflow:
- Digital images typically store pixel intensities using **`np.uint8`** (8-bit unsigned integers), spanning values from `0` (pure black) to `255` (pure white).
- **Overflow/Underflow Hazard:** Because `uint8` handles a maximum value of 255, arithmetic operations can cause silent, disastrous data corruption:
  - In standard NumPy math: `255 + 1 = 0` (Overflow)
  - In standard NumPy math: `0 - 1 = 255` (Underflow)
- Conversely, **OpenCV arithmetic operations** (e.g., `cv2.add()`) implement **saturation arithmetic** where values clip at boundaries: `255 + 1 = 255` and `0 - 1 = 0`. You must keep a strict track of whether you are manipulating variables via raw NumPy operators or OpenCV modules. Some (OpenCV or NumPy) functions will automatically clip (saturate) values, some will allow overflow, and some will automatically upcast (promote) values to a new data type (eg. `np.uint8` to `np.uint16` or higher).

### final remarks before we begin programming: 
1. try to use type annotations in your code. This will help you catch errors and make your code more readable.
2. try to use intuitive variable names. This will make your code more readable and easier to understand (eg use '_rgb', '_bgr', '_hsv' in your image variable name).
3. try to use comments to explain your code. 
4. try to use functions to break your code into smaller, more manageable pieces. 
5. try to use a consistent style for your code.

In [ ]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
# (define the file paths for the images, make sure these files are in the same directory as this notebook!)
# run this cell to set the paths
EAVISE_COLOR_FILE_PATH = "assets/eavise_color.jpeg"
EAVISE_GRAYSCALE_FILE_PATH = "assets/eavise_grayscale.jpeg"

In [ ]:
"""WRITE YOUR CODE HERE"""

# read both the images using OpenCV

# print the dimensions (shape) of the images, what does each dimension represent?

# print the raw data of the images, can you structure in your mind how the data is stored in the array?

# print the data types of the images, are they what you expect?

# display the images using matplotlib (double check the colors match with the actual colors of the images)

# read the images using OpenCV, but use the grayscale flag, what changes?

### 1.2 In-Depth Matrix Slicing & Region of Interest (ROI) Manipulations

Because images are stored as NumPy arrays, extracting parts of an image or isolating color properties uses standard Python slicing syntax: `array[start:end, start:end, ...]`.

- **Slicing Spatial Coordinates (ROIs):** Sub-sampling rows and columns allows you to crop specific spatial patches.
- **Slicing Channel Dimensions:** You can isolate specific spectral colors directly using their index layers (`0` for Blue, `1` for Green, `2` for Red in BGR space).
- **Memory Reference Warning:** Extracting an array slice creates a **view** rather than a hard copy. Modifying the sliced sub-array directly mutates your original image unless you explicitly instantiate a memory duplicate via `.copy()`.
- **Broadcasting:** is NumPy's mechanism for performing arithmetic or assignment operations on arrays with different shapes. It automatically expands the smaller array to match the shape of the larger array. 
    1. Because of this you can for example compare an entire image to a single pixel value, you dont need to create a new array that matches the shape.
    2. You can also broadcast for example a column vector to a matrix, or a row vector to a matrix.

In [ ]:
"""WRITE YOUR CODE HERE"""

# make a copy of the eavise color image variable from the previous cell (to avoid corrupting the original) 
# and crop the upper left quadrant of the image

# fill all pixels of the BLUE channel (make sure you select the right one) 
# with the value 255 using slicing and broadcasting

# display the cropped image using matplotlib 

# you can also display the original eavise color image variable again to check it was not corrupted

# (you can also do the cropping and color-filling by directly referencing the original image variable, 
# to appreciate that the original image will then actually be modified)


### 1.3 The Core Visualizations: Matplotlib BGR vs RGB Pitfall & Grayscale Mapping

Matplotlib's standard display function, `plt.imshow()`, expects images to follow the **RGB** ordering pattern.

1. **Color Interventions:** Passing a raw OpenCV image frame into `plt.imshow()` will swap the spatial mapping of Blue and Red data pipelines, creating severe color inversion artifacting. you can use the `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` function to change the channel ordering (alternatively, you can use the `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)` function to convert the image to grayscale, or `cv2.cvtColor(img, cv2.COLOR_RGB2BGR)` to revert the color ordering).
2. **Grayscale Automatic Scaling Issues:** When passing a 2D array matrix into `plt.imshow()`, Matplotlib automatically applies a default pseudocolor map (Viridis) and automatically scales the color intensity limits to match the minimum and maximum data values found inside the array. To visualize actual physical grayscale data properly, you must manually supply the absolute bounds parameter `vmin=0, vmax=255` alongside the grayscale color mapping configuration (`cmap='gray'`).

In [ ]:
"""WRITE YOUR CODE HERE"""

# plot the eavise color image using matplotlib, with the right channel order

# plot the eavise grayscale image using matplotlib, and use the absolute bounds parameters to see the difference

## Section 2: Advanced Color Spaces & Target Object Segmentation

### 2.1 HSV Transformations, Binary Masks, and API Standards

Segmenting entities using raw RGB thresholds is highly unreliable due to physical lighting intensity shifts variance affecting all three channel vectors simultaneously. 

To counter this, we use the **HSV (Hue, Saturation, Value)** space:
- **Hue (0–179):** Describes the pure chromatic color type (independent of brightness). The standard full circle $360^\circ$ mapping scale is squeezed into 180 entries to fit within `uint8` storage limits.
- **Saturation (0–255):** Quantifies color vividness or purity.
- **Value (0–255):** Quantifies brightness.

to convert from RGB to HSV, we use the `cv2.cvtColor(image, cv2.COLOR_RGB2HSV)` function, the resulting image is still of dtype `uint8` and has shape `(H, W, 3)`. where the channel ordering is now `(H, S, V)`.

#### The Core Functions:
- **`mask = cv2.inRange(src, lowerb, upperb)`**: Scans the input array matrix data against minimum (`lowerb`) and maximum (`upperb`) threshold numpy array boundaries. It yields a **binary mask matrix** where compliant cells map to `255` (True / Active) and out-of-range cells drop to `0` (False / Inactive). The shape remains `(H, W)`, changing the dtype to an 8-bit single-channel representation.
- **`segmented_region = cv2.bitwise_and(src1, src2, mask)`**: Applies an extraction layer. It performs a bit-by-bit logical operation over array indices, keeping only the spatial pixels matching the active binary mask regions.

Note that a (binary) **mask** can be represented in many ways, in this case it is witht the uint8 values `255` and `0`, but it can also be represented as `1` and `0` or as a `bool` array, some functions accept only one of these representations, others are more flexible and accept any of them.

Note that in this case the binary mask is represented with dtype `uint8` and shape `(H, W)`, where the foreground pixels are `255` and the background pixels are `0`. This encoding convention is used to avoid the need for an additional separate background channel. In the case of multi-class segmentation the mask encoding can either be dense (each class has a different value, and each pixel value is no longer binary but rather a class index (can hold up to 255 classes), hence only needing a single channel, containing all class encodings) or sparse/one-hot (where there is a separate channel for each class, and each pixel in the channel is binary, this encoding is less memory efficient as it requires a separate channel for each class, but is sometimes more convenient for analysis and also allows for easy generalization to multi-label setups).

In [ ]:
"""WRITE YOUR CODE HERE"""

# take the eavise color image and convert it to hsv-color space

# define upper and lower bound tresholds for the hue, saturation, and value in the hsv-color space to segment the bright color letters

# apply the thresholding to the image to obtain a binary mask

# display the binary mask with matplotlib

# use the binary mask to segment the image (i.e. extract the foreground and making the background black)

# display the segmented image with matplotlib

In industry, it is common to use these color space segmentation to detect certain objects, for example on a strawberry farm, the color segmentation can be used to detect the strawberries, this is already a very effective first step for downstream applications like object detection, counting, ripeness estimation, etc. Try to detect the strawberries in the image below!

In [ ]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
# (define the file paths for the images, make sure these files are in the same directory as this notebook!)
STRAWBERRIES_FILE_PATH = "assets/strawberries.jpeg"

In [ ]:
"""WRITE YOUR CODE HERE"""


### 2.2 Global Histograms, Automated Otsu Binarization, CLAHE

Now that we've covered manual thresholding, we'll implement automated and adaptive methods for more complex real-world images:
1. **Histogram Equalization (HE):** Enhances local image contrast while preserving global image structure.
2. **Otsu's Binarization:** Automatically calculates the optimal global threshold value by minimizing intra-class variance between the foreground and background of a bimodal histogram (mathematically identical to a 1D Fisher’s Linear Discriminant Analysis (LDA)).
3. **CLAHE (Contrast Limited Adaptive Histogram Equalization):** Breaks images down into local grid tiles to equalize localized contrast variations independently without blowing out noise signals.

In [ ]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
# (define the file paths for the images, make sure these files are in the same directory as this notebook!)
SUPERMARKET_TICKET_FILE_PATH = "assets/ticket.jpeg"

In [ ]:
"""WRITE YOUR CODE HERE"""

# load the ticket image using OpenCV, you can directly pass the grayscale flag to load the image as grayscale

# plot the image using matplotlib

# plot the histogram of the grayscale values using matplotlib (you can use the plt.hist function, dont forget to flatten or ravel the image)

# apply otsu's thresholding to the image using OpenCV

# plot the mask using matplotlib


In [ ]:
"""WRITE YOUR CODE HERE"""

# apply histogram equalization to the image using OpenCV

# plot the histogram of the equalized image using matplotlib

# apply otsu's thresholding to the histogram equalized image using OpenCV

# plot the mask using matplotlib


In [ ]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
# (use this to apply on the image)
clahe_filter_obj = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(100, 100))

In [ ]:
"""WRITE YOUR CODE HERE"""

# apply CLAHE to the image using the filter object from above

# plot the histogram of clahe_img using matplotlib

# apply otsu's thresholding to the clahe image using OpenCV

# plot the mask using matplotlib


## Section 3: Morphological operations

The raw color segmentation results from the previous section can be noisy and difficult to use for downstream applications. If we want to count objects, find contours, or process the mass of an object, we need to clean up the segmentation results first. We can use morphological operations for this. These are shape-based operations performed on binary or grayscale images to remove noise, fill gaps, and smooth out object boundaries. 

### 1. Structuring Elements (Kernels)

Morphological operations apply a small shape called a **Structuring Element** (or kernel) to an image. The kernel slides across the image, comparing its shape with the underlying pixels to determine the output. 

Common shapes include: 

* **Rectangular:** Best for blocky or geometric structures.
* **Elliptical/Circular:** Best for natural, rounded, or organic objects.
* **Cross-shaped:** Useful for analyzing line networks or specific grid alignments.

### 2. Fundamental Operations

All morphological transformations are built on two primary operations: 

* **Erosion:** Shrinks foreground objects by stripping away pixels from their boundaries. A pixel remains a 1 (white) only if *all* pixels under the structuring element are 1. It is highly effective for removing small, isolated noise particles and detaching weakly connected objects.
* **Dilation:** Expands foreground objects by adding pixels to their boundaries. A pixel becomes a 1 if *at least one* pixel under the structuring element is 1. It is ideal for filling in small interior holes and bridging narrow gaps.

### 3. Derived Operations (Compound Operations)

By combining erosion and dilation sequentially, we can achieve more sophisticated cleanup without distorting the overall scale of our objects: 

****Opening****
Erosion followed by Dilation, Removes small bright noise artifacts, breaks thin protrusions, and separates joined shapes while preserving original object sizes.

****Closing****
Dilation followed by Erosion, Fills small dark holes inside objects and bridges narrow gaps/cracks between neighboring elements.

****Morphological Gradient****
Dilation minus Erosion, Outlines the boundaries of your objects, effectively acting as an edge detector for binary shapes.

****Top-Hat****
Original Image minus Opening, Isolates bright elements or features that are smaller than the structuring element against a dark background.

****Black-Hat****
Closing minus Original Image, Isolates dark gaps, holes, or features that are smaller than the structuring element.

### 4.Deep Dive: OpenCV Morphological Functions

To implement these operations in Python, OpenCV provides two highly flexible functions: cv2.getStructuringElement() to design the neighborhood shape, and cv2.morphologyEx() to execute the transformations. 

#### 4.1. Creating Kernels with cv2.getStructuringElement

While you can create a simple rectangular kernel using a NumPy array of ones (np.ones((5,5), np.uint8)), OpenCV provides this function to easily generate non-rectangular neighborhoods like circles or crosses:

*kernel = cv2.getStructuringElement(shape, ksize, anchor)*

Parameters:

1. **shape (Element Shape):** Defines the geometric distribution of the pixel neighborhood. 

  * cv2.MORPH_RECT: A solid rectangular box. Every pixel in the window is considered.
  * cv2.MORPH_ELLIPSE: An ellipse inscribed in the rectangle. Ideal for **circular or organic objects** because it prevents square distortions along edges.
  * cv2.MORPH_CROSS: A cross-shaped element containing only a vertical and horizontal line intersecting at the center.
2. **ksize (Kernel Size):** A tuple (width, height) representing the pixel dimensions of the kernel. 

  * **Note:** It almost always uses **odd numbers** (e.g., (3,3), (5,5), (7,7)) so that the kernel has an unambiguous, perfectly centered pixel.

#### 4.2. Executing Operations with cv2.morphologyEx

Instead of forcing you to call separate functions for every single task, OpenCV routes all advanced, compound morphological transformations through a single master function. 

python

*dst = cv2.morphologyEx(src, op, kernel, iterations)*

Parameters:

1. **src:** The source image. For segmentation cleanup, this is typically a **binary (black and white) mask** of type np.uint8.
2. **op (Operation Flag):** Specifies the mathematical routine to perform: 

  * cv2.MORPH_ERODE: Standard erosion.
  * cv2.MORPH_DILATE: Standard dilation.
  * cv2.MORPH_OPEN: Opening
  * cv2.MORPH_CLOSE: Closing
  * cv2.MORPH_GRADIENT: Outline generation
  * cv2.MORPH_TOPHAT: Bright feature isolation
  * cv2.MORPH_BLACKHAT: Dark feature isolation
3. **kernel:** The structuring element array generated by cv2.getStructuringElement.
4. **iterations (Optional):** The number of times the erosion or dilation sequence is sequentially applied. Default is 1. Increasing iterations amplifies the cleaning effect (e.g., an opening with 2 iterations will erode twice, then dilate twice).

In [ ]:
"""NO NEED TO CHANGE ANYTHING IN THIS CELL!"""
# (define the file paths for the images, make sure these files are in the same directory as this notebook!)
RUBIKSCUBE_CORRUPTED_PATH = "assets/rubiks_cube_corrupted.jpeg"
BARCODE_PATH = "assets/barcode.jpeg"


Segment all the colors of the rubiks cube using the color segmentation method we learned in the previous section. Then, use morphological operations to clean up the segmentation results. Try different kernel sizes and shapes to see how they affect the segmentation results. Visualize the kernel arrays and the cleaned up segmentation results.

In [ ]:
"""WRITE YOUR CODE HERE"""


Someone in the packaging facility has bad intentions and tried to corrupt the barcode on packages. He scratched the barcode, erasing some of the black ink. The barcode is now unreadable to the scanners. Can you use morphological operations to recover the barcode stripes (tip a kernel does not have to be a square shape). the digits do not have to be readable, as long as the vertical stripes are intact to allow scanning.

In [ ]:
"""WRITE YOUR CODE HERE"""
